<div style="padding: 20px; background: linear-gradient(90deg, #1cb5e0 0%, #000851 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🎯 Module 5.3: Maximal Marginal Relevance (MMR)</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Solving the redundancy problem by balancing relevance and diversity.</p>
</div>

---

## 1. The Redundancy Problem
If your document corpus has the same information repeated multiple times (e.g., 5 news articles about the exact same event), a standard semantic search will return all 5 chunks because they are all "closest" to the query.

This wastes the LLM's context window. 

## 2. What is MMR?
Maximal Marginal Relevance (MMR) is an algorithm that fixes this. It:
1. Fetches a large pool of candidates (`fetch_k`, e.g., 20).
2. Selects the single most relevant document.
3. For the remaining documents, it penalizes them if they are too similar to the document already selected.
4. It selects the next document that is both relevant to the query **AND** diverse from the already selected documents.

### The Lambda Multiplier ($\lambda$)
- `lambda_mult = 1.0`: Pure relevance (identical to standard search).
- `lambda_mult = 0.0`: Pure diversity (will pick wildly different documents).

### Course alignment and free-first stack

- Covers: Maximal Marginal Relevance for balancing relevance and diversity.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
embeddings = HuggingFaceEmbeddings(model_name=os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"))

# Notice how the first three sentences contain the EXACT same information.
docs = [
    "Paris is the capital of France.",
    "The capital city of France is Paris.",
    "France's capital is the city of Paris.",
    "London is the capital of England.",
    "Berlin is the capital of Germany.",
]
vs = Chroma.from_documents([Document(page_content=d) for d in docs], embeddings, collection_name="mmr_demo")

query = "Tell me about European capitals."

Let's compare the outputs!

In [ ]:
# 1. Standard Search (Fails due to redundancy)
std = vs.similarity_search(query, k=3)
print("--- Standard Search (k=3) ---")
for d in std:
    print(f"• {d.page_content}")
print("Result: We wasted our entire context window learning that Paris is in France 3 times!\n")

# 2. MMR Search (Succeeds via diversity)
# fetch_k=5 means pull 5 candidates. lambda_mult=0.5 means 50% relevance, 50% diversity.
mmr = vs.max_marginal_relevance_search(query, k=3, fetch_k=5, lambda_mult=0.5)
print("--- MMR Search (k=3, lambda=0.5) ---")
for d in mmr:
    print(f"• {d.page_content}")
print("Result: We learned about France, Germany, and England. Much better for RAG!")